In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
data_path = r'C:\Users\riddh\Downloads\DA-E-Commerce-Data-Challenge-main'
print(f'Data directory: {data_path}')
print(f'Directory exists: {os.path.exists(data_path)}')
print(f'\nContents:')
for item in os.listdir(data_path):
    print(f'  - {item}')


Data directory: C:\Users\riddh\Downloads\DA-E-Commerce-Data-Challenge-main
Directory exists: True

Contents:
  - buyer.csv
  - LICENSE
  - metadata.pdf
  - sales.csv
  - Vendor Datasets


In [3]:
# Cell 3: Load sales.csv
sales = pd.read_csv(os.path.join(data_path, 'sales.csv'))
print(f'Sales data shape: {sales.shape}')
print(f'\nSales data columns: {list(sales.columns)}')
print(f'\nFirst few rows:')
print(sales.head())


Sales data shape: (376004, 6)

Sales data columns: ['order_id', 'buyer_id', 'sku_id', 'quantity', 'order_datetime', 'order_status']

First few rows:
    order_id buyer_id                    sku_id  quantity order_datetime  \
0  O00062060   B10879  PPSTO-PQYYY-E90286460DB4       1.0   7/4/24 13:14   
1  O00186767   B23775  PCLSO-WQEPS-38FA2B8ECC0C       1.0  7/12/25 10:24   
2  O00030197   B27328  PSOCY-NKDLI-31BB2CB8CE81       1.0  3/31/24 10:29   
3  O00110296   B13263  PTGBO-JDTOJ-F5BD3985D601       1.0  11/26/24 7:41   
4  O00085844   B19369  PBEBA-WQEPS-4381545163AF       1.0  9/14/24 19:23   

  order_status  
0    Delivered  
1    Delivered  
2    Delivered  
3    Delivered  
4    Delivered  


In [ ]:
# Cell 4: Clean order_status issues\ntypo_count = (sales['order_status'] == 'Deliverred').sum()\nnull_count = sales['order_status'].isna().sum()\n\nprint(f"Rows with 'Deliverred' before fix: {typo_count}")\nprint(f"Rows with null order_status before drop: {null_count}")\n\n# Issue 3: Standardize misspelled status\nsales['order_status'] = sales['order_status'].replace('Deliverred', 'Delivered')\n\n# Issue 4: Remove rows with missing order status\nsales = sales.dropna(subset=['order_status']).copy()\n\nprint(f"\\nSales shape after cleaning: {sales.shape}")\nprint('Updated order_status value counts:')\nprint(sales['order_status'].value_counts(dropna=False))\n

## Sales EDA - Final Investigation\nReproducible data cleaning, datetime validation, duplicate-grain investigation, and summary metrics for the sales dataset.\n

In [ ]:
# Final cleaning pipeline (non-destructive; creates clean_sales)sales_raw = pd.read_csv(os.path.join(data_path, 'sales.csv'))clean_sales = sales_raw.copy()# Normalize key ID fieldsfor col in ['order_id', 'buyer_id', 'sku_id']:    clean_sales[col] = clean_sales[col].astype('string').str.strip()# Standardize order_status and remove null order_status rowsstatus_typo_count = (clean_sales['order_status'] == 'Deliverred').sum()status_null_count = clean_sales['order_status'].isna().sum()clean_sales['order_status'] = clean_sales['order_status'].replace('Deliverred', 'Delivered')clean_sales = clean_sales.dropna(subset=['order_status']).copy()# Preserve raw datetime, parse strictly first, then fallback parse for unresolved rowsclean_sales['order_datetime_raw'] = clean_sales['order_datetime'].astype('string')clean_sales['order_datetime'] = pd.to_datetime(    clean_sales['order_datetime_raw'],    format='%m/%d/%y %H:%M',    errors='coerce')initial_nat = clean_sales['order_datetime'].isna().sum()try:    import dateparser    def _fallback_parse(x):        if pd.isna(x):            return pd.NaT        dt = dateparser.parse(            str(x).strip(),            settings={'DATE_ORDER': 'MDY', 'PREFER_LOCALE_DATE_ORDER': False}        )        return pd.Timestamp(dt) if dt else pd.NaT    mask_nat = clean_sales['order_datetime'].isna()    clean_sales.loc[mask_nat, 'order_datetime'] = clean_sales.loc[mask_nat, 'order_datetime_raw'].apply(_fallback_parse)except ImportError:    passafter_fallback_nat = clean_sales['order_datetime'].isna().sum()recovered_rows = initial_nat - after_fallback_nat# Keep only valid business date range observed in data (2024-2025)range_invalid_count = ((clean_sales['order_datetime'] < '2024-01-01') | (clean_sales['order_datetime'] > '2025-12-31 23:59:59')).sum()clean_sales = clean_sales[clean_sales['order_datetime'].notna()].copy()clean_sales = clean_sales[    (clean_sales['order_datetime'] >= '2024-01-01') &    (clean_sales['order_datetime'] <= '2025-12-31 23:59:59')].copy()print('CLEANING SUMMARY')print('-' * 70)print(f'Original rows:                          {len(sales_raw):,}')print(f"Order status typo fixed ('Deliverred'):   {status_typo_count:,}")print(f'Null order_status rows removed:         {status_null_count:,}')print(f'Initial datetime NaT after strict parse:{initial_nat:,}')print(f'Recovered via fallback parser:          {recovered_rows:,}')print(f'Remaining datetime NaT removed:         {after_fallback_nat:,}')print(f'Out-of-range datetime rows removed:     {range_invalid_count:,}')print(f'Final clean_sales rows:                 {len(clean_sales):,}')print(f"Date range: {clean_sales['order_datetime'].min()} to {clean_sales['order_datetime'].max()}")

In [ ]:
# Duplicate-grain investigationprint('DUPLICATE INVESTIGATION')print('-' * 70)# Broad duplicate signal (order_id only)dup_order_rows = clean_sales.duplicated(subset=['order_id'], keep=False).sum()# Candidate transaction key used in investigationkey_cols = ['order_id', 'buyer_id', 'sku_id', 'order_datetime']dup_composite_rows = clean_sales.duplicated(subset=key_cols, keep=False).sum()# Scenario A/B check on looser key (order_id+sku_id+datetime)loose_key = ['order_id', 'sku_id', 'order_datetime']loose_dup_mask = clean_sales.duplicated(subset=loose_key, keep=False)scenario_a_rows = (loose_dup_mask & clean_sales.duplicated(keep=False)).sum()loose_dups = clean_sales.loc[loose_dup_mask].copy()if loose_dups.empty:    scenario_b_rows = 0    scenario_b_groups = 0else:    sig = loose_dups.astype(str).agg('|'.join, axis=1)    tmp = loose_dups[loose_key].copy()    tmp['sig'] = sig.values    versions = tmp.groupby(loose_key)['sig'].nunique()    scenario_b_groups = int((versions > 1).sum())    scenario_b_rows = int(loose_dups.set_index(loose_key).index.isin(versions[versions > 1].index).sum())print(f'Rows with duplicate order_id:           {dup_order_rows:,}')print(f'Rows duplicated on strict composite key:{dup_composite_rows:,}')print(f'Scenario A rows (loose key, full copy): {scenario_a_rows:,}')print(f'Scenario B rows (loose key, diff values): {scenario_b_rows:,}')print(f'Scenario B groups (loose key):          {scenario_b_groups:,}')

In [ ]:
# Final EDA summariesprint('FINAL EDA SUMMARY')print('-' * 70)total_rows = len(clean_sales)unique_orders = clean_sales['order_id'].nunique()unique_buyers = clean_sales['buyer_id'].nunique()unique_skus = clean_sales['sku_id'].nunique()avg_items_per_order = total_rows / unique_ordersprint(f'Total rows:             {total_rows:,}')print(f'Unique orders:          {unique_orders:,}')print(f'Unique buyers:          {unique_buyers:,}')print(f'Unique SKUs:            {unique_skus:,}')print(f'Average items/order:    {avg_items_per_order:.2f}')print('\\nOrder status distribution:')status_dist = clean_sales['order_status'].value_counts().to_frame('count')status_dist['pct'] = (status_dist['count'] / total_rows * 100).round(2)print(status_dist)print('\\nQuantity distribution:')print(clean_sales['quantity'].value_counts(dropna=False).sort_index())print('\\nMonthly order volume (head/tail):')monthly = clean_sales['order_datetime'].dt.to_period('M').value_counts().sort_index()print(monthly.head(6))print('...')print(monthly.tail(6))print('\\nTop 10 SKUs by order lines:')print(clean_sales['sku_id'].value_counts().head(10))